### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [3]:
# FAISS: vector store for indexing/retrieving embeddings
from langchain_community.vectorstores import FAISS
# HuggingFaceEmbeddings: creates embeddings from text for semantic search
from langchain_huggingface import HuggingFaceEmbeddings
# TextLoader: loads raw text documents from files
from langchain_community.document_loaders import TextLoader
# RecursiveCharacterTextSplitter: chunks text into smaller pieces for embedding/retrieval
from langchain_text_splitters import RecursiveCharacterTextSplitter
# init_chat_model: initializes the chat model used for query expansion and answer generation
from langchain.chat_models import init_chat_model
# PromptTemplate: builds prompts for the LLM with variables and formatting
from langchain_core.prompts import PromptTemplate
# create_stuff_documents_chain: combines retrieved docs into a single context chain for LLM input
from langchain_classic.chains.combine_documents import create_stuff_documents_chain 
# create_retrieval_chain: creates the end-to-end retrieval chain using retriever + LLM
from langchain_classic.chains.retrieval import create_retrieval_chain
# StrOutputParser: parses model output as a plain string
from langchain_core.output_parsers import StrOutputParser
# RunnableMap: maps inputs to runnable components in a chain or pipeline
from langchain_core.runnables import RunnableMap

/home/valac/Projects/RAG/RAG-Udemi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Step 1: Load and split the data
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks


[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [5]:
# Step 2: Save the chunks in a FAISS vector store with HuggingFace embeddings
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 989.97it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# Step 3: create retriever and LLM for query expansion
# MMR retriever: retrieves relevant documents while maximizing diversity to avoid redundancy
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5})


In [7]:
# Step 4: initialize the LLM and prompt
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
llm = init_chat_model("openai:gpt-5.4-nano")
llm

ChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x72fdce0a7a40>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x72fdbc3a44a0>, root_client=<openai.OpenAI object at 0x72fdf1dd1be0>, root_async_client=<openai.AsyncOpenAI object at 0x72fdce0a7b00>, model_name='gpt-5.4-nano', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [27]:
# step 5: Query Expansion 
# Define the prompt template for query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context. do not include any formatting only plain text .

Original query: "{query}"

Expanded query:                                         
""")

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context. do not include any formatting only plain text .\n\nOriginal query: "{query}"\n\nExpanded query:                                         \n')
| ChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x72fdce0a7a40>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x72fdbc3a44a0>, root_client=<openai.OpenAI object at 0x72fdf1dd1be0>, root_async_client=<openai.AsyncOpenAI object at 0x72fdce0a7b00>, model_name='gpt-5.4-nano', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)
| StrOutputParser()

### query_expansion_chain explain:

The `query_expansion_chain` is a LangChain pipeline that expands user queries for better document retrieval.

#### How the `|` operator works:
- The `|` operator chains LangChain components together in a sequential pipeline.
- Each component processes the output of the previous one, passing data from left to right.
- This creates a "chain" where input flows through each step automatically.

#### Breakdown of each part in the chain:

1. **`query_expansion_prompt` (PromptTemplate)**:
   - Takes the input query and formats it into a structured prompt string.
   - Uses the template defined earlier, replacing `{query}` with the actual user query.
   - Output: A formatted string ready for the LLM.

2. **`llm` (ChatOpenAI model)**:
   - Receives the formatted prompt string from the previous step.
   - Calls the OpenAI API to generate a response using the specified model (gpt-5.4-nano).
   - Output: An AIMessage object containing the LLM's response.

3. **`StrOutputParser()`**:
   - Takes the AIMessage object from the LLM and extracts just the text content.
   - Removes any metadata or formatting, leaving a clean string.
   - Output: Plain string of the expanded query.

#### Chain flow example:
- Input: `"machine learning"`
- Step 1: Formats into prompt → `"You are a helpful assistant. Expand the following query... Original query: "machine learning"..."`
- Step 2: LLM generates → `"machine learning, ML, artificial intelligence, AI, data science, algorithms..."`
- Step 3: Parser extracts → `"machine learning, ML, artificial intelligence, AI, data science, algorithms..."`

This expanded query can then be used for more effective document retrieval in the RAG system.

In [13]:
# Simple Example: How the chain works step by step

# Step 1: Define a simple input query
original_query = "deep learning"

print("=" * 60)
print("QUERY EXPANSION CHAIN EXAMPLE")
print("=" * 60)

# Step 2: Input enters the chain
print(f"\n1️⃣  INPUT: {original_query}")
print(f"   Data type: {type(original_query)}")

# Step 3: Pass through the chain
# This is what happens inside: query_expansion_prompt | llm | StrOutputParser()
print("\n2️⃣  INSIDE THE CHAIN:")
print("   ├─ query_expansion_prompt: Formats query into prompt template")
print("   ├─ llm: Sends prompt to OpenAI and gets response")
print("   └─ StrOutputParser(): Extracts text from LLM response")

# Step 4: Execute the chain and get the expanded query
expanded_query = query_expansion_chain.invoke({"query": original_query})

# Step 5: Output result
print(f"\n3️⃣  OUTPUT: {expanded_query}")
print(f"   Data type: {type(expanded_query)}")

# Step 6: Use the expanded query for retrieval
print("\n4️⃣  USAGE: Pass expanded query to retriever for better document matching")
print(f"   retrieved_docs = retriever.invoke(expanded_query)")
print("=" * 60)

QUERY EXPANSION CHAIN EXAMPLE

1️⃣  INPUT: deep learning
   Data type: <class 'str'>

2️⃣  INSIDE THE CHAIN:
   ├─ query_expansion_prompt: Formats query into prompt template
   ├─ llm: Sends prompt to OpenAI and gets response
   └─ StrOutputParser(): Extracts text from LLM response

3️⃣  OUTPUT: Expanded query:  
“deep learning” OR (neural networks) OR (deep neural network) OR (DNN) OR (multi-layer perceptron) OR (representation learning) OR (feature learning) OR (hierarchical feature learning) OR (end-to-end learning) OR (supervised learning) OR (unsupervised learning) OR (self-supervised learning) OR (semi-supervised learning) OR (contrastive learning) OR (transfer learning) OR (fine-tuning) OR (reinforcement learning) OR (deep reinforcement learning) OR (convolutional neural networks) OR (CNN) OR (deep CNN) OR (image classification) OR (computer vision) OR (object detection) OR (segmentation) OR (recurrent neural networks) OR (RNN) OR (LSTM) OR (GRU) OR (transformers) OR (self-atten

In [15]:
# Step 7: RAG answering Prompt Template
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

In [16]:
# Step 8 : Create Document Chain 
document_chain = create_stuff_documents_chain(llm=llm, prompt=answer_prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
| ChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x72fdce0a7a40>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x72fdbc3a44a0>, root_client=<openai.OpenAI object at 0x72fdf1dd1be0>, root_async_client=<openai.AsyncOpenAI object at 0x72fdce0a7b00>, model_name='gpt-5.4-nano', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [19]:
# Step 9: Create the full RAG Pipeline with query expansion, retrieval, and answer generation
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],  # Step 1: Wrap input query in dict for query expansion
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))  # Step 2: Expand query and retrieve context
    })
    | document_chain  # Step 3: Pass retrieved context to document chain for answer generation
)
rag_pipeline

{
  input: RunnableLambda(...),
  context: RunnableLambda(...)
}
| RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
    context: RunnableLambda(format_docs)
  }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
  | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
  | ChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x72fdce0a7a40>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x72fdbc3a44a0>, root_client=<openai.OpenAI object at 0x72fdf1dd1be0>, root_async_client=<openai.AsyncOpenAI object at 0x72fdce0a7b00>, model_name='gpt-5.4-nano', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)
  | StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[]

### rag_pipeline Explanation:

The `rag_pipeline` is a complete RAG (Retrieval-Augmented Generation) system that combines query expansion, document retrieval, and answer generation.

#### How RunnableMap works:
- `RunnableMap` is a LangChain component that runs multiple operations in parallel and combines their outputs into a dictionary.
- It takes a dictionary where keys are output field names and values are functions (lambdas) that compute those fields.
- All operations run simultaneously, then their results are merged into a single output dictionary.

#### What lambda functions are:
- **Lambda functions** are anonymous, inline functions in Python: `lambda arguments: expression`
- They take an input `x` and return a transformed value.
- Lambdas are used here because `RunnableMap` needs callable functions for each output field.

#### Breakdown of the pipeline:

1. **`RunnableMap` processing (parallel execution)**:
   - **`"input"` field**: `lambda x: x["input"]`
     - Takes the input dictionary `x`
     - Extracts the original query string from the `"input"` key
     - Output: the raw query string, which is passed later to `document_chain`
   
   - **`"context"` field**: `lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))`
     - Takes the input dictionary `x`
     - Reads the original query from `x["input"]`
     - Expands the query with `query_expansion_chain`
     - Uses the expanded query to retrieve relevant documents from the vector store
     - Returns the retrieved document chunks as context

2. **Pipe to `document_chain`**:
   - The output from `RunnableMap` is a dict with `"input"` and `"context"` keys.
   - `document_chain` uses `answer_prompt` and the LLM to generate the final answer.
   - It combines the retrieved `context` with the original `input` question.

#### Data flow example:
- Input: `{"input": "What is machine learning?"}`
- RunnableMap processes in parallel:
  - `"input"`: extracts `"What is machine learning?"`
  - `"context"`: expands the query and retrieves relevant documents
- `document_chain`: uses the question + retrieved context to generate the answer

#### Why this architecture:
- **Query expansion** improves retrieval quality by broadening the search.
- **Parallel field creation** lets the pipeline produce both the input and context together.
- **Modular design** makes it easy to swap or modify components independently.

In [24]:
# Step 10: Run the RAG pipeline with a sample question
query = {"input": "What types of memory does LangChain support?"}
print("=" * 60)
print("Query expansion prompt")
print(query_expansion_prompt.invoke({"query": query}))
print("=" * 60)
print(query_expansion_chain.invoke({"query": query}))
print("=" * 60)
print("Response from Rag pipeline:")
response = rag_pipeline.invoke(query)
print(f"\nRAG Pipeline Response:\n{response}")

Query expansion prompt
text='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{\'input\': \'What types of memory does LangChain support?\'}"\n\nExpanded query:                                         \n'
Expanded query:
```json
{
  "input": "What types of memory does LangChain support? In LangChain memory implementations, what memory backends and modules are available (e.g., conversation buffers, conversation summary, windowed memory, token-based memory, key–value store memory, chat history memory, Redis/Postgres/vector-store-backed memory)? Include supported memory classes/connectors for storing chat context and conversation state, short-term vs long-term memory, and any built-in Memory abstractions (e.g., BufferMemory, ConversationSummaryMemory, ConversationBufferWindowMemory, entity/knowledge memory, retrieval-augmented memory). Also cover integrations for per

In [25]:
query = {"input": "What types of memory does Crew AI support?"}
print("=" * 60)
print("Query expansion prompt")
print(query_expansion_prompt.invoke({"query": query}))
print("=" * 60)
print(query_expansion_chain.invoke({"query": query}))
print("=" * 60)
print("Response from Rag pipeline:")
response = rag_pipeline.invoke(query)
print(f"\nRAG Pipeline Response:\n{response}")

Query expansion prompt
text='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{\'input\': \'What types of memory does Crew AI support?\'}"\n\nExpanded query:                                         \n'
{"input":"CrewAI supported memory types / memory mechanisms / state management in CrewAI. What kinds of memory does CrewAI provide, including short-term vs long-term memory, conversational memory, task/agent state persistence, knowledge storage, context retention, retrieval-augmented memory (RAG), vector store / embeddings-based memory, chat history management, tool/function execution context, and integration with external memory backends (e.g., databases, Redis, vector databases, LangChain memory). Also ask about memory scope per agent vs per crew vs per run, configuration options, and any built-in or pluggable memory components."}
Response from Rag pipeline:

RAG

In [28]:
query = {"input": "CrewAi agents?"}
print("=" * 60)
print("Query expansion prompt")
print(query_expansion_prompt.invoke({"query": query}))
print("=" * 60)
print(query_expansion_chain.invoke({"query": query}))
print("=" * 60)
print("Response from Rag pipeline:")
response = rag_pipeline.invoke(query)
print(f"\nRAG Pipeline Response:\n{response}")

Query expansion prompt
text='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context. do not include any formatting only plain text .\n\nOriginal query: "{\'input\': \'CrewAi agents?\'}"\n\nExpanded query:                                         \n'
CrewAI agents OR CrewAI agent framework OR CrewAI agentic workflow OR CrewAI multi-agent system OR CrewAI orchestration OR CrewAI agent roles OR CrewAI task execution OR CrewAI planning and delegation OR CrewAI tool use OR CrewAI function calling OR LLM agents in CrewAI OR autonomous agents framework OR multi-agent collaboration OR agent swarm OR agent crew OR agent teamwork OR orchestration layer for LLMs OR agent runtime OR agent management OR agent configuration OR CrewAI documentation OR how to use CrewAI agents OR building CrewAI agents with LLMs OR integrating tools and APIs in CrewAI agents OR CrewAI examples OR CrewAI best practices
Resp